<a href="https://colab.research.google.com/github/biswal-prem-5677/Coding-Journey/blob/main/languages/python/libraries/data_analysis_stack/experiment_voice_noise_reduction_and_enhancement_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install noisereduce librosa soundfile numpy

In [6]:
import noisereduce as nr
import librosa
import soundfile as sf

# load audio
audio, sr = librosa.load("/content/asd.wav", sr=None)

# apply noise reduction
reduced_noise = nr.reduce_noise(
    y=audio,
    sr=sr,
    prop_decrease=0.8
)

# save cleaned audio
sf.write("clean_audio.wav", reduced_noise, sr)

print("Noise reduction complete. File saved as clean_audio.wav")

Noise reduction complete. File saved as clean_audio.wav


In [ ]:
# reduced_noise = nr.reduce_noise(
#     y=audio,
#     sr=sr,
#     stationary=False
# )

In [5]:
import librosa
import noisereduce as nr
import soundfile as sf
import numpy as np
from scipy.signal import butter, filtfilt

# -----------------------------
# High-pass filter (remove rumble)
# -----------------------------
def highpass_filter(data, cutoff, sr, order=4):
    nyquist = 0.5 * sr
    normal_cutoff = cutoff / nyquist
    b, a = butter(order, normal_cutoff, btype='high')
    return filtfilt(b, a, data)

# -----------------------------
# Presence boost (voice clarity)
# -----------------------------
def presence_boost(audio, sr):
    # boost frequencies around speech clarity range
    fft = np.fft.rfft(audio)
    freqs = np.fft.rfftfreq(len(audio), 1/sr)

    boost = np.ones_like(freqs)

    boost[(freqs > 2000) & (freqs < 4000)] *= 1.3
    boost[(freqs > 4000) & (freqs < 6000)] *= 1.2

    enhanced_fft = fft * boost
    return np.fft.irfft(enhanced_fft)

# -----------------------------
# Soft compressor
# -----------------------------
def compressor(audio, threshold=0.03, ratio=3):
    compressed = np.copy(audio)

    for i in range(len(audio)):
        if abs(audio[i]) > threshold:
            excess = abs(audio[i]) - threshold
            compressed[i] = np.sign(audio[i]) * (threshold + excess/ratio)

    return compressed

# -----------------------------
# Load audio
# -----------------------------
input_file = "/content/asd.wav"
output_file = "enhanced_voice.wav"

audio, sr = librosa.load(input_file, sr=None)

# -----------------------------
# Step 1 Remove rumble
# -----------------------------
audio = highpass_filter(audio, cutoff=90, sr=sr)

# -----------------------------
# Step 2 Noise reduction
# -----------------------------
audio = nr.reduce_noise(
    y=audio,
    sr=sr,
    stationary=False,
    prop_decrease=0.85
)

# -----------------------------
# Step 3 Vocal clarity boost
# -----------------------------
audio = presence_boost(audio, sr)

# -----------------------------
# Step 4 Compression
# -----------------------------
audio = compressor(audio)

# -----------------------------
# Step 5 Normalize loudness
# -----------------------------
audio = audio / np.max(np.abs(audio))

# -----------------------------
# Save output
# -----------------------------
sf.write(output_file, audio, sr)

print("Voice enhancement complete:", output_file)

Voice enhancement complete: enhanced_voice.wav
